# 14. Paoay: review mapping feasibility and calendar context

**One task:** summarize map/crop outputs within each reference area and compare them with the original notes. This is a feasibility/calendar-context review, not independent local testing accuracy.

No model fitting, prediction, imagery download or automatic ground-truth assignment occurs. Notes and optional calendar interpretations remain unverified. See the [workflow guide](../../docs/paoay_workflow.md).

**Crop-map extent:** only the existing KMZ reference polygons within the study boundary. Outside is masked, not labelled Unknown. Notes remain unverified reference context. The separate Dynamic World map retains the study-wide context.


## 1. Setup

Rerun notebook 13 first for the three-crop + Unknown baseline and separate Dynamic World context. Previous map/report bundles stay intact. Optional `calendar_file` can point to a completed calendar CSV. Otherwise a separate blank template is created once and preserved on reruns. Default outputs already show source notes beside area predictions; calendar entries are not required.


In [ ]:
from pathlib import Path
import json
import sys
repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import review
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress
settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
paoay_root = report_folder.parent / "paoay"
calendar_file = None  # Optional CSV; leave None to use the preserved generated template.
map_pointer = paoay_root / "maps/map_stage.json"
review_root = paoay_root / "review"
review_pointer = review_root / "review_stage.json"
write_stage_pointer(review_pointer, status="pending", reason="Notebook 14 has not finished the current result review")
stages = ["Verify maps and calendar context", "Summarize reference areas", "Review notes and predictions", "Verify review report", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook("14_paoay_review", stages, report_folder.parent / "progress", {"calendar": str(calendar_file)})


## 2. Verify map inputs and optional expectations

Calendar rows use the reference ID, crop name, semicolon-separated month numbers (1–12), scope and an exact excerpt from the original notes. Scopes are `crop_calendar`, `mixed_area`, `seedbed` or `unconfirmed`. Do not infer the historical planting year. Empty rows stay unreviewed; unsupported crops such as lettuce remain outside the three-crop classifier.


In [ ]:
pipeline_progress.begin_stage("Verify maps and calendar context")
map_handoff = resolve_stage_pointer(map_pointer)
request = review.prepare_request(map_handoff, review_root, local_path(calendar_file) if calendar_file else None)
print("Calendar input:", request["calendar_file"])
print("Preserved editable template:", request["calendar_template"])
print("Calendar interpretations describe expectations, not verified crop labels.")
pipeline_progress.end_stage()


## 3. Summarize areas

Review the current three-crop model plus Unknown first. Missing inputs stay separate, and the Dynamic World context maps remain secondary tables. Preserve polygon holes and flag invalid/outside/tiny areas; overlapping areas are not independent tests.


In [ ]:
pipeline_progress.begin_stage("Summarize reference areas")
def area_progress(items):
    return pipeline_progress.track(items, "Review reference areas", unit="area")
review_handoff = review.build(request, iterator=area_progress)
bundle = review.load_review_bundle(review_handoff)
pipeline_progress.end_stage()


## 4. Read notes beside results

Calendar overlap means only that stated months intersect the imagery window. It is not a correctness score, and window-level crop predictions are not dated observations. Mixed-area/seedbed notes remain context only. No accuracy, confusion matrix or F1 is calculated.


In [ ]:
pipeline_progress.begin_stage("Review notes and predictions")
show_table([{**r, "pixels": int(r["pixels"]), "area_ha": float(r["area_ha"]), "roi_percent": float(r["roi_percent"])} for r in bundle["crop_baseline"] if int(r["pixels"]) > 0], title="PRIMARY: three-crop model + Unknown by reference area")
print("Conditional model predictions; not verified crop occupancy or planted area.")
print("Separate Dynamic World/context tables:", bundle["folder"] / "area_composition.csv")
print("Alternative four-class and individual-crop models: TODO only.")
show_table([{key: r[key] for key in ("name", "crop", "calendar_months", "months_overlapping_window", "scope", "status", "window_crop_roi_percent")}
            for r in bundle["calendar"]], title="Calendar context (not accuracy)")
for row in bundle["overview"]:
    print("\n" + row["name"] + " — original unverified notes")
    print(row["original_notes"] or "No source notes.")
pipeline_progress.end_stage()


## 5. Save the review report

The Markdown report and CSV snapshots record original notes, exact map dependencies and calendar interpretations. Changed calendar edits create a new bundle; previous reports and editable calendars are not overwritten.


In [ ]:
pipeline_progress.begin_stage("Verify review report")
review.load_review_bundle(review_handoff)
write_stage_pointer(review_pointer, review_handoff)
print("Readable report:", bundle["folder"] / "report.md")
print("Reference-area tables:", bundle["folder"])
print("Independent local accuracy evaluated: no")
pipeline_progress.end_stage()


## 6. Summary

The implemented 07–14 workflow is complete as a demonstration. Dated crop observations and verified crop-cycle timing would be needed for a separately designed independent local evaluation. Future annotations do not automatically become test labels.

TODO: compare a separately trained pooled four-class model and an individual-crop model grouped for reporting. Both remain deferred. The current model, labels and partitions are unchanged.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Verified review handoff:", review_handoff)
print("Current review pointer:", review_pointer)
print("Outcome: attributed maps, coverage, reference-area summaries and unverified calendar context.")
print("Model and input data preserved. No next implementation batch starts automatically.")
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
